<a href="https://colab.research.google.com/github/RobertLopez893/naturalLanguageProcessing/blob/main/Practica3/tf-idf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Session 3. Text Representation
# López Reyes José Roberto. 7CM2.

## Imports

In [ ]:
import urllib.request
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Fetching the books

In [ ]:
def get_and_clean_gutenberg_book(book_id):
    """Fetches a book from Project Gutenberg and cleanly slices out the fluff."""
    url = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
    try:
        text = urllib.request.urlopen(url).read().decode('utf-8')
    except Exception as e:
        print(f"Error: {e}")
        return ""

    end = re.search(r'\*\*\* END OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*', text)
    if end:
        text = text[:end.start()]

    if book_id == 2591:  # Grimms' Fairy Tales
        start_phrase = "THE GOLDEN BIRD\r\n\r\n\r\nA certain king"
        start_index = text.find(start_phrase)
        if start_index != -1:
            text = text[start_index:]

    elif book_id == 2892:  # Irish Fairy Tales
        start_phrase = "THE STORY OF TUAN MAC CAIRILL\r\n\r\n\r\n\r\nCHAPTER I"
        start_index = text.find(start_phrase)
        if start_index != -1:
            text = text[start_index:]

    return text

## Splitting the book into stories

In [ ]:
def split_into_stories(text):
    """
    Attempts to split the full book text into individual stories or chapters.
    Gutenberg formatting varies wildly, so we split by large blocks of empty space
    or fully capitalized titles, which usually indicate a new story.
    """
    # Regex looks for 3 or more line breaks, or capitalized titles
    chunks = re.split(r'\n\s*\n\s*\n', text)

    # Filter out tiny chunks (like the Table of Contents or intro paragraphs)
    # We only want substantial text chunks (e.g., more than 1000 characters)
    stories = [chunk.strip() for chunk in chunks if len(chunk.strip()) > 1000]
    return stories

## Test

In [ ]:
# Fetch the data
grimms_text = get_and_clean_gutenberg_book(2591)
irish_text = get_and_clean_gutenberg_book(2892)

# Parse into stories
print("Parsing texts into stories...")
grimms_stories = split_into_stories(grimms_text)
irish_stories = split_into_stories(irish_text)

print(f"Found {len(grimms_stories)} story chunks in Grimms' Fairy Tales.")
print(f"Found {len(irish_stories)} story chunks in Irish Fairy Tales.")

# Text Representation (TF-IDF)
print("\nVectorizing text using TF-IDF...")
# We filter out English stop words automatically
vectorizer = TfidfVectorizer(stop_words='english')

# We fit the vectorizer on ALL stories combined so they share the same vocabulary "dictionary"
all_stories = grimms_stories + irish_stories
vectorizer.fit(all_stories)

# Transform the texts into numerical matrix vectors
grimms_vectors = vectorizer.transform(grimms_stories)
irish_vectors = vectorizer.transform(irish_stories)

# Calculate Similarity
print("Calculating similarities...")
# This creates a matrix comparing every Grimms story to every Irish story
similarity_matrix = cosine_similarity(grimms_vectors, irish_vectors)

# Extract the Top 5 most similar pairs
results = []
for i in range(len(grimms_stories)):
    for j in range(len(irish_stories)):
        results.append({
            "Grimms' Chunk Index": i,
            "Irish Chunk Index": j,
            "Similarity Score": similarity_matrix[i][j]
        })

# Convert to a Pandas DataFrame for a nice table view
df = pd.DataFrame(results)

# Sort by the highest similarity score and grab the top 5
top_5 = df.sort_values(by="Similarity Score", ascending=False).head(5)

print("\n" + "="*50)
print("TOP 5 MOST SIMILAR STORIES (Text Representation)")
print("="*50)
print(top_5.to_string(index=False))

Parsing texts into stories...
Found 67 story chunks in Grimms' Fairy Tales.
Found 91 story chunks in Irish Fairy Tales.

Vectorizing text using TF-IDF...
Calculating similarities...

TOP 5 MOST SIMILAR STORIES (Text Representation)
 Grimms' Chunk Index  Irish Chunk Index  Similarity Score
                   9                 10          0.193774
                  64                 90          0.188113
                  62                 90          0.186590
                  34                 18          0.184095
                  34                 90          0.180942


## Verification

In [ ]:
import numpy as np

# Let's inspect your #1 match
g_idx = 9
i_idx = 10

print("==================================================")
print(f" TEXT PREVIEW: WHY DID THEY MATCH?")
print("==================================================")

# Print the first 700 characters of each story to read them
print("\n--- GRIMMS' STORY CHUNK 65 (Preview) ---")
print(grimms_stories[g_idx][:700] + "...\n")

print("--- IRISH STORY CHUNK 90 (Preview) ---")
print(irish_stories[i_idx][:700] + "...\n")

# Let's ask the TF-IDF Vectorizer what words they have in common
print("\n==================================================")
print(" THE 'SMOKING GUN' WORDS")
print("==================================================")
print("These are the shared words with the highest combined TF-IDF scores.\n")

# Get the actual English words from the vectorizer's dictionary
feature_names = vectorizer.get_feature_names_out()

# Grab the numerical scores for our two specific stories
g_scores = grimms_vectors[g_idx].toarray()[0]
i_scores = irish_vectors[i_idx].toarray()[0]

# Find the indices of words where BOTH stories have a score greater than 0
shared_word_indices = np.where((g_scores > 0) & (i_scores > 0))[0]

# Calculate how much each shared word contributed to that 0.188 score
shared_words = []
for idx in shared_word_indices:
    # Multiplying them simulates how Cosine Similarity overlaps vectors
    combined_weight = g_scores[idx] * i_scores[idx]
    shared_words.append((feature_names[idx], combined_weight))

# Sort the words from most important to least important
shared_words.sort(key=lambda x: x[1], reverse=True)

# Print the top 15 words driving the match
print(f"{'Shared Word':<15} | {'Match Weight'}")
print("-" * 35)
for word, weight in shared_words[:15]:
    print(f"{word:<15} | {weight:.5f}")

 TEXT PREVIEW: WHY DID THEY MATCH?

--- GRIMMS' STORY CHUNK 65 (Preview) ---
There was once a fisherman who lived with his wife in a pigsty, close
by the seaside. The fisherman used to go out all day long a-fishing; and
one day, as he sat on the shore with his rod, looking at the sparkling
waves and watching his line, all on a sudden his float was dragged away
deep into the water: and in drawing it up he pulled out a great fish.
But the fish said, ‘Pray let me live! I am not a real fish; I am an
enchanted prince: put me in the water again, and let me go!’ ‘Oh, ho!’
said the man, ‘you need not make so many words about the matter; I will
have nothing to do with a fish that can talk: so swim away, sir, as soon
as you please!’ Then he put him back into the water...

--- IRISH STORY CHUNK 90 (Preview) ---
CHAPTER XI

“THE fisherman of Cairill, the King of Ulster, took me in his net. Ah,
that was a happy man when he saw me! He shouted for joy when he saw the
great salmon in his net.

“I was 